# Pipeline-aren demo: gertaera bera muturretik muturrera (offline)

Koaderno honek CG20 ebidentzia jasoa erakusten du zerbitzu bizirik gabe:
gertaera gordin bat (`raw.csv`) eta erantzun kanoniko bera HTTP/APIan,
ledger-ean, worker-ean, Elasticsearch-en eta Kibana panelean/Discover-en.
Egiaztapena ez da hemen berridazten: `scripts.check_cg20_same_event`
script bera exekutatzen da, eta ondoren kontabilitatea eta lotura
erakusten dira irakurgarri.

Estaldura: fitxategi→ledger→API→worker→ES→panel kate osoa, kontabilitatea
(5 = 4 gorde + 1 ezeztatu + 0 zain), offset-a (5) eta identitate-iragazki
negatiboa. Ez dira Kafka/NiFi biziak ordezkatzen.


## 1. Ingurunea eta egiaztapen osoa

Script ofizialak baieztapen guztiak egiten ditu (kanonikoa berdina bost
lekutan, ES identitatea bakarra, raw SHA-256, panel/Discover balioak,
iragazki negatiboa). Huts egiten badu, ziegiak huts egiten du.


In [1]:
import sys
from pathlib import Path

ROOT = (
    Path.cwd()
    if (Path.cwd() / "data/ai4i2020_train.csv").is_file()
    else Path.cwd().parent
)
print("ROOT:", ROOT)

import subprocess

result = subprocess.run(
    [sys.executable, "-m", "scripts.check_cg20_same_event"],
    cwd=ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(result.stdout.strip())


ROOT: /home/tears/bigdata/01_Erronka1_CNC_Guard/B-taldea
CG20 same-event evidence verified: 279947a017e5d70adba2a83e5eb319c0c7495260ebac7259de1954bb834c6b8b HTTP=ledger=worker=ES; Kibana panel/Discover; negative filter; 5=4+1+0


## 2. Kontabilitatea: 5 = 4 + 1 + 0

Iturri automatikoaren kontabilitatea: 5 behatu, 4 gorde, 1 ezeztatu
(kalitate-arauengatik, trazabeharrekin) eta 0 zain; `committed_offset` 5.
Ezeztapena ez da galera: bereiz bereizten da eta zenbatzen da.


In [2]:
import json

EVIDENCE = ROOT / "docs/evidence/CG20_SAME_EVENT_20261022"
evidence = json.loads((EVIDENCE / "same-event.json").read_text(encoding="utf-8"))

accounting = evidence["accounting"]
print("kontabilitatea:", accounting)
print("committed_offset:", evidence["committed_offset"])
assert accounting == {"observed": 5, "saved": 4, "rejected": 1, "pending": 0}
assert evidence["committed_offset"] == 5
assert accounting["saved"] + accounting["rejected"] + accounting["pending"] == accounting["observed"]
print("OK: 5 = 4 gorde + 1 ezeztatu + 0 zain.")


kontabilitatea: {'observed': 5, 'saved': 4, 'rejected': 1, 'pending': 0}
committed_offset: 5
OK: 5 = 4 gorde + 1 ezeztatu + 0 zain.


## 3. Erantzun kanoniko bera bost lekutan

HTTP/API, ledger, worker eta ES `_source` erantzunak berdin-berdinak
dira. Hona gertaeraren nortasuna eta inferentziaren emaitza.


In [3]:
def read(name):
    return json.loads((EVIDENCE / name).read_text(encoding="utf-8"))


canonical = evidence["canonical"]
assert canonical == read("api-response.json") == read("es-source.json")
assert canonical == evidence["canonical_ledger"] == evidence["worker_response"]
assert canonical == evidence["es_hit"]["_source"]
print("HTTP = ledger = worker = ES: OK")
print("event_id:", canonical["event_id"])
print("machine_id:", canonical["machine_id"], "| timestamp:", canonical["timestamp"])
print("alert_level:", canonical["alert_level"], "| failure_score:", canonical["failure_score"])
print("anomaly_alert:", canonical["anomaly_alert"], "| model_version:", canonical["model_version"])
print("ES index:", evidence["es_hit"]["_index"], "| ES id:", evidence["es_hit"]["_id"])


HTTP = ledger = worker = ES: OK
event_id: 279947a017e5d70adba2a83e5eb319c0c7495260ebac7259de1954bb834c6b8b
machine_id: CNC-001 | timestamp: 2026-10-21T08:00:00Z
alert_level: CRITICA | failure_score: 0.8
anomaly_alert: True | model_version: synthetic-replay-only
ES index: cnc-alerts-automatic-v4 | ES id: a7f78d005fefd2883d6f762aa3befb9c9962083000c333a5d6740c9ee86e1a1c


## 4. Jatorri gordina eta hatz-marka

Sarrera gordinaren lerroa eta bere SHA-256 hatz-marka. Lerroa iturri
fitxategian (`lab-01/input/normal.csv`) dagoela ere baieztatzen du
script-ak; hemen lerroaren aurrebista erakusten da.


In [4]:
import hashlib

raw = (EVIDENCE / "raw.csv").read_bytes().removesuffix(b"\n")
digest = hashlib.sha256(raw).hexdigest()
print("raw SHA-256 bat dator:", digest == evidence["raw_sha256"])
print("lerro gordina:", raw.decode("utf-8")[:160])


raw SHA-256 bat dator: True
lerro gordina: 2026-10-21T08:00:00Z,CNC-001,H,300,310,1500,40,10,1,0,0,0,0,0


## 5. Ebidentzia-fitxategiak eta panelaren balio egiaztatuak

Direktorio jasoa zerrendatzen da, eta Kibana panelean egiaztatutako
balioen (`dashboard-verified-values.json`) lagin bat erakusten da.
Discover-ek errenkada bakarra erakusten du, eta iragazki negatiboak
("0"*64) ez du emaitzarik ematen: identitate-kontsulta zehatza da.


In [5]:
files = sorted(path.name for path in EVIDENCE.iterdir())
print(f"fitxategiak ({len(files)}):")
for name in files:
    print(" -", name)

values = read("dashboard-verified-values.json")
print("\npaneleko balio egiaztatuak:", len(values))
for entry in values[:5]:
    print(" -", entry)


fitxategiak (55):
 - after.log
 - api-instrumentation-repair.log
 - api-response.json
 - before.log
 - checksums.json
 - collect.log
 - commands.jsonl
 - dashboard-negative.ax.txt
 - dashboard-negative.png
 - dashboard-verified-values.json
 - dashboard-verified.ax.txt
 - dashboard-verified.png
 - discover-document.ax.txt
 - discover-document.png
 - discover-negative.ax.txt
 - discover-negative.png
 - discover-restored.ax.txt
 - discover-restored.png
 - discover-same-event.ax.txt
 - discover-same-event.png
 - discover-visible-row.json
 - es-final-pipeline.json
 - es-final-template.json
 - es-mapping.json
 - es-operations.jsonl
 - es-query.json
 - es-source.json
 - export.log
 - kibana-configure-r2.log
 - kibana-configure-r3.log
 - kibana-configure.log
 - kibana-final-runtime-export.ndjson
 - kibana-operations.jsonl
 - kibana-search.log
 - kibana-start-r2.log
 - kibana-start-r3.log
 - kibana-start.log
 - lab-01
 - lab-finish.log
 - lab-normal-r2.log
 - lab-normal.log
 - lab-resume-nifi.l

## 6. Katearen irakurketa eta mugak

- **Katea:** lerro gordina → normalizazioa/baliozkotze → scoring API →
  worker bideraketa → ES idazketa → panel/Discover. Urrats bakoitzean
  `event_id` bera lotzen da.
- **Ezeztapen trazagarria:** kalitate-arauak betetzen ez dituen
  erregistroa ez da isilean galtzen; `rejected` kontagailuan agertzen da.
- **Offset-a:** 5 konprometituta; pausaren/berrabiaraztearen jarraitutasuna
  CG15/CG32 ebidentzian probatzen da, ez ziega honetan.
- **Muga:** ebidentzia jasoa da, ez zerbitzu bizien kontrako proba;
  Kafka/NiFi/ES biziak laborategiko konposizioarekin erakusten dira.

Birsorkuntza (tresna efimeroekin, lock-a aldatu gabe):

```bash
uv run --frozen --with ipykernel --with nbconvert jupyter nbconvert \
  --to notebook --execute notebooks/04_pipeline_ledger_demo.ipynb \
  --ExecutePreprocessor.kernel_name=python3 \
  --output 04_pipeline_ledger_demo.ipynb --output-dir outputs/notebooks-exec
```
